In [28]:
import requests
import pandas as pd
import numpy as np

In [29]:
# Download the json from github
url = "https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/refs/heads/master/json/countries.json"
# Use requests
r = requests.get(url)
r.json()

[{'id': 1,
  'name': 'Afghanistan',
  'iso3': 'AFG',
  'iso2': 'AF',
  'numeric_code': '004',
  'phonecode': '93',
  'capital': 'Kabul',
  'currency': 'AFN',
  'currency_name': 'Afghan afghani',
  'currency_symbol': '؋',
  'tld': '.af',
  'native': 'افغانستان',
  'population': 43844000,
  'gdp': None,
  'region': 'Asia',
  'region_id': 3,
  'subregion': 'Southern Asia',
  'subregion_id': 14,
  'nationality': 'Afghan',
  'area_sq_km': 647500,
  'postal_code_format': None,
  'postal_code_regex': None,
  'timezones': [{'zoneName': 'Asia/Kabul',
    'gmtOffset': 16200,
    'gmtOffsetName': 'UTC+04:30',
    'abbreviation': 'AFT',
    'tzName': 'Afghanistan Time'}],
  'translations': {'br': 'Afghanistan',
   'ko': '아프가니스탄',
   'pt-BR': 'Afeganistão',
   'pt': 'Afeganistão',
   'nl': 'Afghanistan',
   'hr': 'Afganistan',
   'fa': 'افغانستان',
   'de': 'Afghanistan',
   'es': 'Afganistán',
   'fr': 'Afghanistan',
   'ja': 'アフガニスタン',
   'it': 'Afghanistan',
   'zh-CN': '阿富汗',
   'tr': 'Afganist

In [30]:
# Extract the iso3 code, country name, latitude, and longitude
countries = [{"iso3": x.get("iso3"), "currency": x.get("currency")
                , "name": x.get("name"), "latitude": x.get("latitude")
                , "longitude": x.get("longitude")} 
             for x in r.json()]

countries_df = pd.DataFrame(countries)

countries_df["latitude"] = pd.to_numeric(countries_df["latitude"], errors="coerce")
countries_df["longitude"] = pd.to_numeric(countries_df["longitude"], errors="coerce")

countries_df

,iso3,currency,name,latitude,longitude
0,AFG,AFN,Afghanistan,33.000000,65.0
1,ALA,EUR,Aland Islands,60.116667,19.9
2,ALB,ALL,Albania,41.000000,20.0
3,DZA,DZD,Algeria,28.000000,3.0
4,ASM,USD,American Samoa,-14.333333,-170.0
...,...,...,...,...,...
245,WLF,XPF,Wallis and Futuna Islands,-13.300000,-176.2
246,ESH,MAD,Western Sahara,24.500000,-13.0
247,YEM,YER,Yemen,15.000000,48.0
248,ZMB,ZMW,Zambia,-15.000000,30.0


In [31]:
countries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iso3       250 non-null    str    
 1   currency   250 non-null    str    
 2   name       250 non-null    str    
 3   latitude   250 non-null    float64
 4   longitude  250 non-null    float64
dtypes: float64(2), str(3)
memory usage: 9.9 KB


In [32]:
# Calculate distance between countries using the Haversine formula
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

In [35]:
# Get all pairs possible
from itertools import product
from tqdm import tqdm

pairs = list(product(countries_df["iso3"].unique(), repeat=2))

distances = []
for iso3_1, iso3_2 in tqdm(pairs):
    lat1 = countries_df[countries_df["iso3"] == iso3_1]["latitude"].values[0]
    lon1 = countries_df[countries_df["iso3"] == iso3_1]["longitude"].values[0]
    lat2 = countries_df[countries_df["iso3"] == iso3_2]["latitude"].values[0]
    lon2 = countries_df[countries_df["iso3"] == iso3_2]["longitude"].values[0]
    try:
        distance = haversine(lat1, lon1, lat2, lon2)
    except Exception as e:
        distance = np.nan
        print(f"Error calculating distance between {iso3_1} and {iso3_2}: {e}")
    distances.append({"iso3_i": iso3_1, "iso3_j": iso3_2, "distance_km": distance})

distances_df = pd.DataFrame(distances)
distances_df

 33%|███▎      | 20539/62500 [00:44<02:22, 294.69it/s]

Error calculating distance between GHA and TUV: math domain error


 92%|█████████▏| 57759/62500 [02:04<00:09, 495.65it/s]

Error calculating distance between TUV and GHA: math domain error


100%|██████████| 62500/62500 [02:15<00:00, 459.60it/s]


,iso3_i,iso3_j,distance_km
0,AFG,AFG,0.000000
1,AFG,ALA,4436.559161
2,AFG,ALB,4047.637966
3,AFG,DZA,5881.179067
4,AFG,ASM,14114.459916
...,...,...,...
62495,ZWE,WLF,15332.596720
62496,ZWE,ESH,6792.169129
62497,ZWE,YEM,4361.029497
62498,ZWE,ZMB,555.974633


In [36]:
# Save to clean csv
file_path = "../Clean/countries_distances.csv"
distances_df.to_csv(file_path, index=False)